In [87]:
# Standard libraries
import warnings

# Data processing
import numpy as np
import pandas as pd
import json

# PyTorch
import torch

# PostgreSQL and pgvector
import psycopg
from pgvector.psycopg import register_vector

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Re-ranking
from sentence_transformers import CrossEncoder

# LLM
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# Notebook settings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.8.0+cu129
Device: cpu


## Connect to PostgreSQL

In [3]:
# Connect to PostgreSQL
connection = psycopg.connect(
    dbname="postgres",
    user="postgres",
    password="root",
    host="localhost",
    port="5432"
)

cursor = connection.cursor()

register_vector(connection)

print("Connected to PostgreSQL.")

Connected to PostgreSQL.


In [4]:
# Initialize embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6297.80it/s]


In [5]:
# Initialize reranker
reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device=DEVICE
)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 3703.05it/s]


## Run a semantic Search

In [10]:
#connection.rollback()

In [9]:
query_embedding = embedding_model.embed_query(query)

cursor.execute(
    """
    SELECT
        id,
        document_id,
        chunk_id,
        heading_path,
        content,
        embedding <=> %s::vector AS distance
    FROM chunks
    ORDER BY embedding <=> %s::vector
    LIMIT 10;
    """,
    (query_embedding, query_embedding)
)

results = cursor.fetchall()

In [11]:
for index, result in enumerate(results, start=1):
    print("====================")
    print("RESULT", index)
    print("Chunk ID:", result[2])
    print("Path:", result[3])
    print("Distance:", round(float(result[5]), 3))
    print()
    print(result[4])
    print()

RESULT 1
Chunk ID: 12
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Assessment guidelines > Volume minimums and transaction data thresholds
Distance: 0.349

Source: adblue-and-def.md
Context: METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Assessment guidelines > Volume minimums and transaction data thresholds

#### Volume minimums and transaction data thresholds  
Argus typically does not establish thresholds strictly on the basis of a count of transactions, as this could lead to unreliable and non-representative assessments and because of the varying transportation infrastructure found in all commodity markets. Instead, minimum volumes are typically established which may apply to each transaction accepted, to the aggregate of transactions, to transactions which set a low or high assessment or to other volumetrically relevant parameters.  
For price assessments used to settle derivatives, Argus will seek to establish minimum transaction data thresho

## Reranker

In [12]:
# Prepare reranker pairs
pairs = []

for result in results:
    content = result[4]
    pairs.append([query, content])

rerank_scores = reranker.predict(pairs)

In [13]:
# Inspect reranker scores
for index, score in enumerate(rerank_scores, start=1):
    print("Result", index, "| Score:", round(float(score), 3))

Result 1 | Score: 0.016
Result 2 | Score: 0.984
Result 3 | Score: 0.001
Result 4 | Score: 0.0
Result 5 | Score: 0.0
Result 6 | Score: 0.002
Result 7 | Score: 0.0
Result 8 | Score: 0.0
Result 9 | Score: 0.002
Result 10 | Score: 0.0


In [14]:
# Sort reranked results
top_indices = np.argsort(rerank_scores)[::-1][:5]

for index in top_indices:
    result = results[index]
    score = rerank_scores[index]

    print("Chunk ID:", result[2])
    print("Score:", round(float(score), 3))
    print("Path:", result[3])
    print()

Chunk ID: 22
Score: 0.984
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Product specification > Diesel exhaust fluid (DEF)

Chunk ID: 12
Score: 0.016
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Assessment guidelines > Volume minimums and transaction data thresholds

Chunk ID: 23
Score: 0.002
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Spot pricing > US

Chunk ID: 5
Score: 0.002
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Verification of transaction data > Primary tests applied by reporters

Chunk ID: 17
Score: 0.001
Path: ARGUS AMERICAS BIOFUELS > Methodology overview > Volume minimums and transaction data thresholds



In [73]:
# Decompose query
def decompose_query(query):
    decomposer = ChatOllama(model="qwen2.5:1.5b", temperature=0)

    prompt = """
You are a query decomposition component for a RAG system.

Return ONLY a valid JSON array of strings.

Rules:
- If the user query contains only one information request, return the original query exactly.
- If it contains multiple information requests, split it into independent retrieval queries.
- If the query compares, contrasts, or asks for differences between topics, create one retrieval query for each topic.
- Each retrieval query must be understandable independently.
- Preserve all important entities and context in every subquery.
- Repeat entity names when necessary.
- Do not use pronouns if the entity can be stated explicitly.
- Do not answer the query.
- Do not explain anything.
- Do not add commentary.
- Return only valid JSON.

Examples:

Input:
What is DEF?

Output:
["What is DEF?"]

Input:
What is DEF and what is the minimum transaction size?

Output:
["What is DEF?", "What is the minimum transaction size for DEF?"]

Input:
Compare DEF product specification with US spot pricing methodology.

Output:
["What is the DEF product specification?", "What is the US spot pricing methodology for DEF?"]

Input:
""" + query + """

Output:
"""

    response = decomposer.invoke(prompt)

    subqueries = json.loads(response.content)
    if len(subqueries) == 0:
        subqueries = [query]

    return subqueries

## Final function: Retrieve and rerank documents

In [74]:
# Retrieve and rerank documents
def retrieve_and_rerank(query, retrieve_k=10, rerank_k=5, threshold=0.1):
    subqueries = decompose_query(query)

    if len(subqueries) == 1:
        subqueries = [query]

    all_results = []

    for subquery in subqueries:
        query_embedding = embedding_model.embed_query(subquery)

        cursor.execute(
            """
            SELECT
                c.id,
                c.document_id,
                c.chunk_id,
                c.heading_path,
                c.content,
                d.filename,
                c.embedding <=> %s::vector AS distance
            FROM chunks c
            JOIN documents d ON c.document_id = d.id
            ORDER BY c.embedding <=> %s::vector
            LIMIT %s;
            """,
            (query_embedding, query_embedding, retrieve_k)
        )

        results = cursor.fetchall()

        for result in results:
            all_results.append(result)

    unique_results = []
    seen_ids = []

    for result in all_results:
        chunk_db_id = result[0]

        if chunk_db_id not in seen_ids:
            seen_ids.append(chunk_db_id)
            unique_results.append(result)

    pairs = []

    for document in unique_results:
        pairs.append([query, document[4]])

    rerank_scores = reranker.predict(pairs)

    top_indices = np.argsort(rerank_scores)[::-1][:rerank_k]

    if rerank_scores[top_indices[0]] < threshold:
        return "No info"

    top_results = []

    for index in top_indices:
        score = float(rerank_scores[index])

        top_results.append({
            "document": unique_results[index],
            "score": score
        })

    return top_results

## Evaluate

In [75]:
# Define evaluation queries
evaluation_queries = [
    {
        "query": "What is the minimum transaction size for DEF?",
        "relevant_source": "adblue-and-def.md",
        "relevant_path": "Diesel exhaust fluid (DEF)"
    },
    {
        "query": "How is DEF defined?",
        "relevant_source": "adblue-and-def.md",
        "relevant_path": "Diesel exhaust fluid (DEF)"
    },
    {
        "query": "How are transaction data verified?",
        "relevant_source": "adblue-and-def.md",
        "relevant_path": "Verification of transaction data"
    },
    {
        "query": "What primary tests are applied by reporters?",
        "relevant_source": "adblue-and-def.md",
        "relevant_path": "Primary tests applied by reporters"
    },
    {
        "query": "Who won the 2018 FIFA World Cup?",
        "relevant_source": None,
        "relevant_path": None
    }
]

In [76]:
# Evaluate retrieval + reranking
evaluation_results = []

for item in evaluation_queries:
    query = item["query"]
    relevant_source = item["relevant_source"]
    relevant_path = item["relevant_path"]

    top_documents = retrieve_and_rerank(query)

    if top_documents == "No info":
        retrieved_results = "No info"

    else:
        retrieved_results = []

        for result in top_documents:
            document = result["document"]

            retrieved_results.append({
                "source": document[5],
                "path": document[3]
            })

    evaluation_results.append({
        "query": query,
        "relevant_source": relevant_source,
        "relevant_path": relevant_path,
        "retrieved": retrieved_results
    })

In [77]:
# Display evaluation results
for result in evaluation_results:
    print("Query:", result["query"])
    print("Relevant source:", result["relevant_source"])
    print("Relevant path:", result["relevant_path"])
    print("Retrieved:", result["retrieved"])
    print()

Query: What is the minimum transaction size for DEF?
Relevant source: adblue-and-def.md
Relevant path: Diesel exhaust fluid (DEF)
Retrieved: [{'source': 'adblue-and-def.md', 'path': 'METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Product specification > Diesel exhaust fluid (DEF)'}, {'source': 'adblue-and-def.md', 'path': 'METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Assessment guidelines > Volume minimums and transaction data thresholds'}, {'source': 'adblue-and-def.md', 'path': 'METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Spot pricing > US'}, {'source': 'adblue-and-def.md', 'path': 'METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Verification of transaction data > Primary tests applied by reporters'}, {'source': 'americas-biofuels.md', 'path': 'ARGUS AMERICAS BIOFUELS > Methodology overview > Volume minimums and transaction data thresholds'}]

Query: How is DEF defined?
Relevant source: adblue-and-def.md
Releva

In [78]:
# Calculate evaluation metrics
precision_scores = []
recall_scores = []
mrr_scores = []

rejection_correct = 0
rejection_total = 0

for result in evaluation_results:
    relevant_source = result["relevant_source"]
    relevant_path = result["relevant_path"]
    retrieved = result["retrieved"]

    if relevant_source is None:
        rejection_total += 1

        if retrieved == "No info":
            rejection_correct += 1

        continue

    correct = 0
    reciprocal_rank = 0

    for rank, item in enumerate(retrieved, start=1):
        source_match = item["source"] == relevant_source
        path_match = relevant_path in item["path"]

        if source_match and path_match:
            correct += 1

            if reciprocal_rank == 0:
                reciprocal_rank = 1 / rank

    precision = correct / len(retrieved)
    recall = 1 if correct > 0 else 0

    precision_scores.append(precision)
    recall_scores.append(recall)
    mrr_scores.append(reciprocal_rank)

rejection_accuracy = rejection_correct / rejection_total

print("Precision@5:", round(np.mean(precision_scores), 3))
print("Recall@5:", round(np.mean(recall_scores), 3))
print("MRR:", round(np.mean(mrr_scores), 3))
print("Rejection Accuracy:", round(rejection_accuracy, 3))

Precision@5: 0.35
Recall@5: 1.0
MRR: 1.0
Rejection Accuracy: 1.0


In [79]:
# Test query
def test_query(query, threshold=0.1):
    top_results = retrieve_and_rerank(query)

    if top_results == "No info":
        print("No relevant information found.")
        return

    for result in top_results:
        if result["score"] >= threshold:
            document = result["document"]
            score = result["score"]

            print("Chunk ID:", document[2])
            print("Source:", document[5])
            print("Score:", round(score, 3))
            print("Path:", document[3])
            print()

In [80]:
q = "What is DEF?"
test_query(q)

Chunk ID: 22
Source: adblue-and-def.md
Score: 0.988
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Product specification > Diesel exhaust fluid (DEF)

Chunk ID: 21
Source: adblue-and-def.md
Score: 0.863
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU



In [81]:
q = "What is DEF and what is the minimum transaction size?"

test_query(q)

Chunk ID: 22
Source: adblue-and-def.md
Score: 0.986
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Product specification > Diesel exhaust fluid (DEF)

Chunk ID: 21
Source: adblue-and-def.md
Score: 0.294
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU

Chunk ID: 12
Source: adblue-and-def.md
Score: 0.205
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Methodology overview > Assessment guidelines > Volume minimums and transaction data thresholds

Chunk ID: 23
Source: adblue-and-def.md
Score: 0.172
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Spot pricing > US



In [82]:
q = "Compare the DEF product specification with the US spot pricing methodology."

test_query(q)

Chunk ID: 23
Source: adblue-and-def.md
Score: 0.993
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Spot pricing > US

Chunk ID: 27
Source: adblue-and-def.md
Score: 0.385
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Spot pricing > Benelux (Belgium/Netherlands/Luxembourg)

Chunk ID: 22
Source: adblue-and-def.md
Score: 0.347
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Product specification > Diesel exhaust fluid (DEF)

Chunk ID: 25
Source: adblue-and-def.md
Score: 0.237
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > Argus AdBlue®-DEF and TGU > Spot pricing > Asia

Chunk ID: 0
Source: adblue-and-def.md
Score: 0.165
Path: METHODOLOGY AND SPECIFICATIONS GUIDE > ARGUS ADBLUE®-DEF AND TGU > Contents:



## LLM --> final generator

In [90]:
# Initialize final LLM
llm = ChatOllama(model="qwen2.5:7b", temperature=0)

In [91]:
# Build context from retrieved chunks
def build_context(top_results):
    context = ""

    for result in top_results:
        document = result["document"]

        source = document[5]
        path = document[3]
        content = document[4]

        context += "Source: " + source + "\n"
        context += "Path: " + path + "\n"
        context += content + "\n\n"

    return context

In [122]:
# Generate answer
def generate_answer(query):
    cached_answer = check_cache(query)

    if cached_answer is not None:
        print("Cache hit.")
        return cached_answer

    top_results = retrieve_and_rerank(query)

    if top_results == "No info":
        return "No relevant information found."

    context = build_context(top_results)

    prompt = """
Answer the user question using only the provided context.

Rules:
- Do not use outside knowledge.
- If the context does not contain enough information, say that the information is not available.
- Be concise and accurate.
- Do not invent facts.

Context:
""" + context + """

Question:
""" + query + """

Answer:
"""

    response = llm.invoke(prompt)

    answer = response.content
    sources = []

    for result in top_results:
        document = result["document"]
        source = document[5]
    
        if source not in sources:
            sources.append(source)
    
    answer += "\n\nSources: " + ", ".join(sources)

    save_cache(query, answer)

    return answer

## Semantic Cache

In [123]:
# Check semantic cache
def check_cache(query, threshold=0.05):
    query_embedding = embedding_model.embed_query(query)

    cursor.execute(
        """
        SELECT
            answer,
            query_embedding <=> %s::vector AS distance
        FROM semantic_cache
        ORDER BY query_embedding <=> %s::vector
        LIMIT 1;
        """,
        (query_embedding, query_embedding)
    )

    result = cursor.fetchone()

    if result is None:
        return None

    if result[1] <= threshold:
        return result[0]

    return None

In [124]:
# Save answer to semantic cache
def save_cache(query, answer):
    query_embedding = embedding_model.embed_query(query)

    cursor.execute(
        """
        INSERT INTO semantic_cache (
            query,
            query_embedding,
            answer
        )
        VALUES (%s, %s, %s);
        """,
        (
            query,
            query_embedding,
            answer
        )
    )

    connection.commit()

In [131]:
# "What is the minimum DEF transaction size?" --> don't want cache hit
# "What is the maximum transaction size for DEF?" --> don't want cache hit
# "What is the smallest DEF transaction size?" --> I want cache hit
#q = "What is the maximum transaction size for DEF?"
q = "How is DEF defined?"
answer = generate_answer(q)
print(answer)

Cache hit.
DEF (Diesel exhaust fluid) is defined by the ISO-22241 standard as an aqueous urea solution with a urea content of 32.5% by weight.

Sources: adblue-and-def.md


In [132]:
q = "What is the minimum volume for Argo ethanol in Chicago?"
answer = generate_answer(q)
print(answer)

The minimum volume for Argo ethanol in Chicago is 3,000 bl.

Sources: americas-biofuels.md


In [133]:
q = "What are the specifications for B100 and B99 biodiesel?"
answer = generate_answer(q)
print(answer)

B100: biodiesel with 100pc purity. Conforms to ASTM D6751  
B99: biodiesel with 99pc purity. Conforms to ASTM D6751

Sources: americas-biofuels.md


In [134]:
q = "What is a RIN and how can it be traded after renewable fuel is blended?"
answer = generate_answer(q)
print(answer)

A RIN, or Renewable Identification Number, is generated upon the production or import of renewable fuels. After a gallon of renewable fuel is blended, the RIN may be detached from the physical fuel and sold through the EPA Moderated Tracking System (EMTS). Refiners, importers, and blenders can purchase RINs to fulfill government mandates set forth by RFS2.

Sources: americas-biofuels.md


In [135]:
q = "What is the minimum aggregate volume required for the Argo ethanol volume-weighted average?"
answer = generate_answer(q)
print(answer)

The minimum aggregate volume required for the Argo ethanol volume-weighted average is 15,000 bl.

Sources: americas-biofuels.md


In [137]:
q = "Compare the minimum transaction size for DEF with the minimum volume for Argo ethanol in Chicago."
answer = generate_answer(q)
print(answer)

The minimum transaction size for DEF is 1,500 US liquid gallons (approximately 5,680 litres) or more. For Argo ethanol in Chicago, the minimum volume is 3,000 barrels (bl).

Sources: adblue-and-def.md, americas-biofuels.md


In [138]:
q = "What are the specifications for DEF and for B100 biodiesel?"
answer = generate_answer(q)
print(answer)

The specifications for DEF include:
- It is an aqueous urea solution defined by ISO-22241 with a 32.5% urea content by weight.
- Automotive-grade urea, which can be in solid or liquid form (aqueous urea solution with a urea content ≥ 40%), is combined with deionised water.

For B100 biodiesel:
- It has 100% purity and conforms to ASTM D6751.
- The cloud point for B100 varies by location throughout the year, with maximum values of 2.0°C, 8.0°C, or 13.0°C depending on the start and end dates provided in the table.

Sources: americas-biofuels.md, adblue-and-def.md
